# U9-01 · Datos grandes: Parquet, una sola pasada, DuckDB y MapReduce

*Machine Learning e Inteligencia Artificial (IFCD0174) · Unidad 9, apartados 9.1 a 9.5*

Ejecuta cada celda con **Mayús + Intro**. Para que funcione en el navegador, la flota simulada tiene **50 máquinas** en lugar de 200 (438 000 lecturas en lugar de 1 752 000): los tamaños y tiempos serán menores que en el libro y los resultados, parecidos pero no idénticos.

In [ ]:
%pip install -q duckdb pyarrow
import duckdb, pyarrow
print("DuckDB", duckdb.__version__, "· PyArrow", pyarrow.__version__)

## 0. Los datos simulados

El generador de la unidad (archivo `lecturas.py` del libro), con 50 máquinas. Cada hora registra temperatura, vibración, consumo y horas desde la revisión; desde octubre, la mitad de la flota mide la temperatura 3 °C de más.

In [ ]:
import numpy as np
import pandas as pd

def generar_lecturas(maquinas=50, horas=24 * 365, semilla=0):
    """Lecturas horarias simuladas de una flota durante un año."""
    rng = np.random.default_rng(semilla)
    ritmo = rng.uniform(0.5, 1.5, maquinas)      # desgaste por hora
    desgaste = rng.uniform(0, 300, maquinas)     # desde la revisión
    filas = []
    for h in range(horas):
        desgaste += ritmo
        temp = 60 + 0.01 * desgaste + rng.normal(0, 2, maquinas)
        if h >= 24 * 273:              # desde octubre, media flota
            temp[: maquinas // 2] += 3  # con sensor descalibrado
        vib = 2 + 0.004 * desgaste + rng.normal(0, 0.5, maquinas)
        kw = 40 + 0.5 * (temp - 60) + rng.normal(0, 3, maquinas)
        z = -9 + 0.006 * desgaste + 0.8 * (vib - 2)
        averia = rng.random(maquinas) < 1 / (1 + np.exp(-z))
        filas.append((np.full(maquinas, h), np.arange(maquinas),
                      temp, vib, kw, desgaste.copy(), averia))
        desgaste[averia] = 0                      # se repara
    h, m, t, v, k, d, a = map(np.concatenate, zip(*filas))
    df = pd.DataFrame({
        "fecha": pd.Timestamp("2025-01-01") + pd.to_timedelta(h, "h"),
        "maquina": m, "temperatura": t, "vibracion": v,
        "consumo_kw": k, "horas_desde_revision": d, "averia": a})
    # objetivo: ¿se averiará la máquina en las próximas 24 horas?
    df = df.sort_values(["maquina", "fecha"], ignore_index=True)
    def en_24h(s):                               # ¿avería en las 24 h
        futuro = s[::-1].rolling(24, 1).max()[::-1]   # siguientes?
        return futuro.shift(-1, fill_value=0)
    proxima = df.groupby("maquina")["averia"].transform(en_24h)
    df["averia_24h"] = proxima.astype(bool)
    return df

## 1. Memoria, tipos y formatos (ejemplo 1)

In [ ]:
import os, time
import pandas as pd

df = generar_lecturas()                  # 50 máquinas, un año
mb = lambda d: d.memory_usage(deep=True).sum() / 2**20
print(f"{len(df):,} filas, {mb(df):.1f} MB en memoria")

# tipos más pequeños: mismo contenido, menos memoria
df = df.astype({"maquina": "int16", "temperatura": "float32",
                "vibracion": "float32", "consumo_kw": "float32",
                "horas_desde_revision": "float32"})
print(f"con tipos ajustados: {mb(df):.1f} MB")

for ruta, guardar in [("lecturas.csv", df.to_csv),
                      ("lecturas.parquet", df.to_parquet)]:
    t0 = time.perf_counter()
    guardar(ruta, index=False)
    seg = time.perf_counter() - t0
    print(f"{ruta:<17} {os.path.getsize(ruta) / 2**20:6.1f} MB"
          f"  escrito en {seg:.2f} s")

os.makedirs("particiones", exist_ok=True)   # un archivo por mes
for mes, trozo in df.groupby(df["fecha"].dt.month):
    trozo.to_parquet(f"particiones/mes={mes:02d}.parquet",
                     index=False)

## 2. Leer CSV y Parquet

In [ ]:
import time
import pandas as pd

def cronometrar(leer):
    t0 = time.perf_counter()
    leer()
    return time.perf_counter() - t0

lecturas = {
  "CSV completo": lambda: pd.read_csv("lecturas.csv",
                                      parse_dates=["fecha"]),
  "Parquet completo": lambda: pd.read_parquet("lecturas.parquet"),
  "Parquet, 2 columnas": lambda: pd.read_parquet(
      "lecturas.parquet", columns=["maquina", "averia"])}
for nombre, leer in lecturas.items():
    print(f"{nombre:<20} {cronometrar(leer):.3f} s")

**Prueba.** Lee el CSV solo con `usecols=["maquina", "averia"]`. ¿Se acerca al Parquet? ¿Por qué no del todo?

## 3. Welford y muestreo de embalse

In [ ]:
import numpy as np
import pandas as pd

def welford(flujo):
    """Media y desviación típica en una sola pasada."""
    n, media, m2 = 0, 0.0, 0.0
    for x in flujo:
        n += 1
        d = x - media
        media += d / n
        m2 += d * (x - media)
    return n, media, (m2 / (n - 1)) ** 0.5

def embalse(flujo, k, rng):
    """Muestra aleatoria de k elementos de un flujo de longitud
    desconocida (algoritmo R)."""
    muestra = []
    for i, x in enumerate(flujo):
        if i < k:
            muestra.append(x)
        else:
            j = rng.integers(i + 1)          # al azar entre 0 e i
            if j < k:
                muestra[j] = x
    return muestra

def vibraciones(ruta):                       # lee el CSV por trozos
    for trozo in pd.read_csv(ruta, usecols=["vibracion"],
                             chunksize=100_000):
        yield from trozo["vibracion"].to_numpy()

n, media, sd = welford(vibraciones("lecturas.csv"))
print(f"{n} lecturas: media {media:.4f}, desviación {sd:.4f}")
rng = np.random.default_rng(0)
m = embalse(vibraciones("lecturas.csv"), 1000, rng)
print(f"muestra de {len(m)}: media {np.mean(m):.4f}, "
      f"desviación {np.std(m, ddof=1):.4f}")

In [ ]:
import pandas as pd
v = pd.read_parquet("lecturas.parquet", columns=["vibracion"])["vibracion"]
print(f"con la columna entera: media {v.mean():.4f}, desviación {v.std():.4f}")

**Prueba.** Repite el muestreo con otras semillas (`default_rng(1)`, `(2)`…) y observa cuánto varía la media de la muestra.

## 4. Consultas con DuckDB (ejemplo 2)

In [ ]:
import duckdb

consulta = """
    SELECT strftime(fecha, '%Y-%m')   AS mes,
           count(*)                   AS lecturas,
           count_if(averia)::INTEGER  AS averias,
           round(avg(vibracion), 3)   AS vibracion_media
    FROM 'lecturas.parquet'
    GROUP BY mes
    ORDER BY mes
"""
resumen = duckdb.sql(consulta).df()
print(resumen)

**Ejercicio.** Escribe una consulta que devuelva las 5 máquinas con mayor vibración media, con `GROUP BY maquina ORDER BY ... DESC LIMIT 5`.

## 5. MapReduce en Python

In [ ]:
from collections import Counter
from functools import reduce
import glob
import pandas as pd

def mapear(ruta):
    """Fase map: cada partición cuenta sus averías por máquina."""
    p = pd.read_parquet(ruta, columns=["maquina", "averia"])
    return Counter(p.loc[p["averia"], "maquina"].tolist())

def reducir(a, b):
    """Fase reduce: se suman los recuentos parciales."""
    return a + b

particiones = sorted(glob.glob("particiones/mes=*.parquet"))
parciales = map(mapear, particiones)      # en Spark, en paralelo
total = reduce(reducir, parciales, Counter())
print(len(particiones), "particiones,",
      sum(total.values()), "averías")
print("más averías:", total.most_common(3))

**Ejercicio.** Cambia `mapear` para contar lecturas (no averías) por máquina y comprueba que todas tienen 8760.

> Welford, B. P. (1962). Note on a method for calculating corrected sums of squares and products. *Technometrics*, 4(3), 419–420. https://doi.org/10.1080/00401706.1962.10490022
>
> Vitter, J. S. (1985). Random sampling with a reservoir. *ACM Transactions on Mathematical Software*, 11(1), 37–57. https://doi.org/10.1145/3147.3165
>
> Dean, J. y Ghemawat, S. (2008). MapReduce: simplified data processing on large clusters. *Communications of the ACM*, 51(1), 107–113. https://doi.org/10.1145/1327452.1327492